# Cat Attribute Labeler · Colab Pro+

`Qwen3-VL-32B-Instruct` NF4 4bit로 외부 고양이 사진 최대 500장을 2단계 라벨링합니다.
런타임 → 런타임 유형 변경에서 **A100 GPU와 고용량 RAM**을 우선 선택하세요.
GPU는 가용성에 따라 달라집니다. 최소 21GiB free VRAM/BF16 및 85GiB local free disk를 실제 점검합니다.
T4 16GB는 이 설정으로 실행할 수 없습니다. L4 24GB는 GPU smoke에서 확인해야 합니다.

위에서 아래로 실행하세요. Drive 권한 승인은 직접 진행합니다.
다운로드 → 실제 2장 smoke → 통과하면 500장 처리가 이어집니다.
동일 RUN_NAME을 여러 서버/노트북에서 동시에 실행하지 마세요.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
REPO = Path('/content/cat-attribute-labeler')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/jeehoo0507/cat-attribute-labeler.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
subprocess.run(['nvidia-smi'], check=True)


## 저장 경로

새 작업은 새 RUN_NAME을 사용합니다. 재개할 때는 이전과 같은 RUN_NAME을 유지하세요.
사진·카탈로그·단계별 라벨은 Drive 하위 폴더에 저장합니다.
모델과 Python 환경은 빠른 런타임 로컬 디스크에 저장하며, 런타임이 삭제되면 다시 다운로드합니다.
Drive에 파일이 많아지면 I/O가 느려질 수 있어 기본 500장 파일럿에 사용하세요.


In [ ]:
RUN_NAME = 'colab32b_pilot_v1'
PERSIST = Path('/content/drive/MyDrive/cat-attribute-labeler')
os.environ.update({
    'DATA_ROOT': str(PERSIST / 'data'),
    'OUTPUT_ROOT': str(PERSIST / 'outputs' / RUN_NAME),
    'HF_HOME': '/content/cat-labeler-hf',
    'LOCK_ROOT': '/content/cat-labeler-locks',
    'GPUS': '0',
    'CONFIG': 'configs/pilot.json',
    'RUN_NAME': RUN_NAME,
    'PUBLISH_RESULTS': '0',
})
for name in ['DATA_ROOT', 'OUTPUT_ROOT', 'HF_HOME', 'LOCK_ROOT']:
    Path(os.environ[name]).mkdir(parents=True, exist_ok=True)
print('Persistent annotations:', os.environ['OUTPUT_ROOT'])


## 실행 환경 설치

독립 `.venv`를 사용합니다. Colab 커널에 모델 의존성을 설치하지 않으므로 커널 재시작이 필요하지 않습니다.


In [ ]:
subprocess.run(['bash', './setup.sh'], check=True)


## GPU 확인 후 전체 파이프라인 실행

이 셀의 실행을 유지하세요. `nohup`으로 우회하지 않고 노트북 실행 작업으로 처리합니다.
실패하면 다음 단계로 넘어가지 않습니다. 오류를 해결한 뒤 같은 셀을 다시 실행하면 처리한 항목은 재사용합니다.
코드가 변경된 새 revision을 사용해 재실행할 때는 프롬프트/설정 서명 검사를 확인하세요.
Pro+도 충분한 compute units가 있어야 최대 24시간 연속 실행이 가능하며 GPU는 보장되지 않습니다.

실행이 중단되어도 Drive에 저장 완료된 라벨과 데이터는 재사용할 수 있습니다.
출력 라벨은 전부 검수 전 초안입니다.


In [ ]:
def execute_pipeline():
    log_path = Path(os.environ['OUTPUT_ROOT']) / 'colab.log'
    process = subprocess.Popen(
        ['bash', './run_server.sh'], stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True,
    )
    try:
        with log_path.open('a', encoding='utf-8') as log:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
        code = process.wait()
        if code:
            raise RuntimeError(f'Pipeline stopped with exit code {code}. Log: {log_path}')
    except BaseException:
        import signal
        import time
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    print('Completed. Results remain annotation drafts requiring review.')

execute_pipeline()


## 상태 확인 또는 일부 결과 내보내기

전체 실행이 끝나거나 실행 중지를 누른 뒤 사용하세요.


In [ ]:
subprocess.run(['bash', './status.sh'], check=True)


## 결과 ZIP을 Drive에 저장

중단된 실행은 `--allow-partial`로 완료된 라벨만 내보냅니다. summary.json에 완료 여부가 기록됩니다.


In [ ]:
import shutil
subprocess.run([
    '.venv/bin/python', '-m', 'cat_labeler.export',
    '--name', RUN_NAME, '--allow-partial',
], check=True)
archive = shutil.make_archive(f'/content/{RUN_NAME}', 'zip', root_dir=REPO / 'results' / RUN_NAME)
destination = PERSIST / 'exports' / Path(archive).name
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, destination)
print('Saved:', destination)


## 선택 사항 · GitHub 결과 푸시

Colab 왼쪽 열쇠 아이콘의 Secrets에 `GH_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요.
해당 레포 Contents 쓰기 권한이 있는 토큰을 사용합니다. 토큰을 코드 셀에 붙여 넣지 마세요.
`PUSH_TO_GITHUB=True`로 바꾸고 이 셀을 실행하면 결과 bundle만 현재 브랜치에 commit/push합니다.
Drive의 원본 사진·모델 가중치는 Git에 추가되지 않습니다.


In [ ]:
PUSH_TO_GITHUB = False
if PUSH_TO_GITHUB:
    from google.colab import userdata
    import requests
    token = userdata.get('GH_TOKEN')
    response = requests.get('https://api.github.com/user', headers={'Authorization': f'Bearer {token}'}, timeout=20)
    response.raise_for_status()
    account = response.json()
    subprocess.run(['apt-get', 'update', '-qq'], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'gh'], check=True, stdout=subprocess.DEVNULL)
    environment = {**os.environ, 'GH_TOKEN': token}
    subprocess.run(['gh', 'auth', 'setup-git'], env=environment, check=True)
    subprocess.run(['git', 'config', 'user.name', account['login']], check=True)
    subprocess.run(['git', 'config', 'user.email', f"{account['id']}+{account['login']}@users.noreply.github.com"], check=True)
    subprocess.run(['bash', './publish_results.sh'], env=environment, check=True)
    del token, environment
else:
    print('GitHub push skipped. Results are saved on Drive.')


실행 제한과 Drive 저장 관련 공식 안내 — https://research.google.com/colaboratory/faq.html
